In [ ]:
#Filter based on Cluster_id with each class

import pandas as pd

df = pd.read_csv("seq_cluster_class_final.csv", sep=";")

# cluster size per class
summary = df.groupby(["label", "Cluster_id"]).size().reset_index(name="cluster_size")

order = ["Sec", "Tat", "T1SS", "T2SS", "T3SS", "T4SS", "T5SS", "T6SS", "Non-secreted"]
summary["label"] = pd.Categorical(summary["label"], categories=order, ordered=True)

# sort biggest cluster first, per class (easier manual bin-packing, place big piece first)
summary = summary.sort_values(["label", "cluster_size"], ascending=[True, False])

# running total + running percent, PER CLASS
summary["running_total(Cumsum)"] = summary.groupby("label")["cluster_size"].cumsum()
summary["class_total_seq"] = summary.groupby("label")["cluster_size"].transform("sum")
summary["Percentage_seq_cover"] = (summary["running_total(Cumsum)"] / summary["class_total_seq"] * 100).round(1)

# suggested split — greedy, first ~20% of running total goes test, rest train (you can override manual)
summary["Train/Test_Split"] = summary["Percentage_seq_cover"].apply(lambda p: "train" if p <= 80 else "test")

summary["final_split"] = summary["Train/Test_Split"] 

summary.to_csv("cluster_split_decision.csv", index=False, sep=";")
print("done, total cluster rows:", len(summary))

done, total cluster rows: 3330


In [ ]:
#Train /Test Split based on Percentage of Sequence cover
# Assign label in cluster_split_decision.csv file

import pandas as pd

# load real sequences from fasta
seqs = {}
with open("combine_final.fasta") as f:
    header, seq_lines = None, []
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            if header:
                seqs[header] = "".join(seq_lines)
            header = line[1:]
            seq_lines = []
        else:
            seq_lines.append(line)
    if header:
        seqs[header] = "".join(seq_lines)

# load full sequence-level data + cluster split decision
df = pd.read_csv("seq_cluster_class_final.csv", sep=";")
decision = pd.read_csv("cluster_split_decision.csv", sep=";")

# attach split decision to every sequence via Cluster_id
merged = df.merge(decision[["Cluster_id", "final_split"]], on="Cluster_id", how="left")

# attach real protein sequence
merged["Sequence"] = merged["Protein_id"].map(seqs)

train_df = merged[merged["final_split"] == "train"]
test_df = merged[merged["final_split"] == "test"]

train_df.to_csv("train_final.csv", index=False, sep=";")
test_df.to_csv("test_final.csv", index=False, sep=";")

print("Train size:", len(train_df))
print("Test size:", len(test_df))
print("\nTrain distribution:\n", train_df["label"].value_counts())
print("\nTest distribution:\n", test_df["label"].value_counts())

overlap = set(train_df["Cluster_id"]) & set(test_df["Cluster_id"])
print("\nCluster overlap (must be 0):", len(overlap))

Train size: 2927
Test size: 736

Train distribution:
 label
Non-secreted    1353
T4SS             446
T3SS             357
Sec              262
T6SS             220
T1SS             118
Tat               76
T2SS              64
T5SS              31
Name: count, dtype: int64

Test distribution:
 label
Non-secreted    339
T4SS            113
T3SS             89
Sec              66
T6SS             55
T1SS             30
Tat              20
T2SS             16
T5SS              8
Name: count, dtype: int64

Cluster overlap (must be 0): 1


In [ ]:

# Create .fasta file of Train and Test from .csv file
import pandas as pd

train_df = pd.read_csv("train_final.csv", sep=";")
test_df = pd.read_csv("test_final.csv", sep=";")

def write_fasta(df, filename):
    with open(filename, "w") as f:
        for _, row in df.iterrows():
            f.write(f">{row['Protein_id']}\n{row['Sequence']}\n")

write_fasta(train_df, "train.fasta")
write_fasta(test_df, "test.fasta")

print("train.fasta lines:", sum(1 for _ in open("train.fasta")))
print("test.fasta lines:", sum(1 for _ in open("test.fasta")))

train.fasta lines: 5854
test.fasta lines: 1472
